### 프롬프트 
- AI 모델과의 상호작용을 표준화하고 효울적으로 만들어 주는 구조화 된 텍스트 형식
- 프롬프트 템플릿: 지시 사항(Instruction), 질문(Question), 검색된 정보인 문맥 (Context)로 구성
- from_templeate() 메서드를 사용하여 PromptTemplate 객체 생성 및 객체 생성과 동시에 프롬프트 생성 방법 있음

In [3]:
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv()
logging.langsmith("CH02-Prompt")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH02-Prompt


In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    temperature=0,
    model_name = 'gpt-4o-mini'
)

In [6]:
from langchain_core.prompts import PromptTemplate

template = "{country}의 수도는 어디인가요?"

prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='{country}의 수도는 어디인가요?')

In [7]:
prompt = prompt.format(country = '대한민국')
prompt

'대한민국의 수도는 어디인가요?'

In [11]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 1. 프롬프트 정의
template = "{country}의 수도는 어디인가요?"
prompt = PromptTemplate.from_template(template)

# 2. 모델 및 파서 생성
llm = ChatOpenAI(model="gpt-4o-mini")

# 3. 체인(Chain) 연결
chain = prompt | llm

# 4. 실행 (입력값은 딕셔너리 형태!)
chain.invoke("대한민국").content

'대한민국의 수도는 서울입니다.'

In [13]:
#객체에서 바로 실행하는 과정

template = "{country}의 수도는 어디인가요?"

prompt = PromptTemplate(
    template=template,
    input_variables=["country"]
)

prompt.format(country = '대한민국')

'대한민국의 수도는 어디인가요?'

In [16]:
#partial_variables로 여러 개의 값 중 미리 채우기

template = "{country1}과 {country2}의 수도는 각각 어디인가요?"

prompt = PromptTemplate(
    template=template,
    input_variables=["country1"],
    partial_variables={
        "country2" : "미국"
    },
)

prompt.format(country1 = "대한민국")

'대한민국과 미국의 수도는 각각 어디인가요?'

In [19]:
#partial 메서드로 템플릿의 일부 변수 채우기 가능
#format() 메서드로 남은 변수 채워넣기 가능

prompt_partial = prompt.partial(country2 = "캐나다")
prompt_partial.format(country1 = "대한민국")

'대한민국과 캐나다의 수도는 각각 어디인가요?'

In [20]:
chain = prompt_partial | llm
chain.invoke("대한민국").content

'대한민국의 수도는 서울이고, 캐나다의 수도는 오타와입니다.'

In [21]:
#덮어 쓰기 가능 (캐나다 -> 호주)

chain.invoke({'country1': '대한민국', 'country2': '호주'}).content

'대한민국의 수도는 서울이고, 호주의 수도는 캔버라입니다.'

부분 변수 활용하기

In [25]:
#현재 날짜를 자동으로 반환하는 함수를 사용하여 프롬프트 설정

from datetime import datetime

def get_today():
    return datetime.now().strftime('%B %d')

prompt = PromptTemplate(
    template='오늘의 날짜는 {today}입니다. 오늘이 생일인 유명인 {n}명을 나열해 주세요. 생년월일을 표기해주세요.',
    input_variables= ["n"],
    partial_variables={
        "today" : get_today
    },
)

prompt.format(n=3)

'오늘의 날짜는 August 01입니다. 오늘이 생일인 유명인 3명을 나열해 주세요. 생년월일을 표기해주세요.'

In [31]:
chain = prompt | llm
print(chain.invoke(3).content)

오늘이 August 01인 유명인 3명은 다음과 같습니다:

1. **버지니아 울프 (Virginia Woolf)** - 1882년 1월 25일
2. **하퍼 리 (Harper Lee)** - 1926년 4월 28일
3. **미셸 오바마 (Michelle Obama)** - 1964년 1월 17일

고맙습니다!


In [34]:
print(chain.invoke({'today': 'Jan 02', "n": 3}).content)

1. **한니발 바르카 (Hannibal Barca)** - 247 BC, 1월 2일
2. **신사임당 (Shin Saimdang)** - 1504년 1월 2일
3. **지미 로저스 (Jimmy Rogers)** - 1897년 1월 2일

이 외에도 유명한 인물들이 있지만, 생일이 같은 이들을 몇 명 소개해드렸습니다.


YAML 파일로부터 프롬프트 템플릿 로드

In [41]:
from langchain_core.prompts import load_prompt

prompt = load_prompt('../Prompt/fruit_color.yaml', encoding='utf-8')
prompt

PromptTemplate(input_variables=['fruit'], input_types={}, partial_variables={}, template='{fruit}의 색깔이 뭐야?')

In [42]:
prompt.format(fruit = '사과')

'사과의 색깔이 뭐야?'

In [44]:
prompt2 = load_prompt('../Prompt/capital.yaml')
print(prompt2.format(country = '대한민국'))

대한민국의 수도에 대해서 알려주세요.
수도의 특징을 다음의 양식에 맞게 정리해 주세요.
300자 내외로 작성해 주세요.
한글로 작성해 주세요.
----
[양식]
1. 면적
2. 인구
3. 역사적 장소
4. 특산품

#Answer:



In [46]:
from langchain_core.output_parsers import StrOutputParser
from langchain_teddynote.messages import stream_response

chain = prompt2 | ChatOpenAI(model_name ='gpt-4o', temperature=0) | StrOutputParser()

answer = chain.stream({"country" : "대한민국"})
stream_response(answer)

1. 면적: 서울특별시는 약 605.21㎢의 면적을 가지고 있으며, 이는 대한민국의 수도로서 다양한 행정, 경제, 문화 활동이 이루어지는 중심지입니다.  
2. 인구: 서울의 인구는 약 950만 명으로, 대한민국에서 가장 인구가 많은 도시입니다. 다양한 인종과 문화가 공존하는 국제적인 도시로 성장하고 있습니다.  
3. 역사적 장소: 경복궁, 창덕궁, 덕수궁 등 조선시대의 궁궐들이 있으며, 한양도성, 종묘 등 유네스코 세계문화유산으로 지정된 역사적 장소들이 많습니다.  
4. 특산품: 서울은 전통과 현대가 조화를 이루는 도시로, 한복, 한지 공예품, 전통 음식인 김치와 떡 등이 유명합니다. 또한, 현대적인 패션과 기술 제품도 특산품으로 꼽힙니다.

ChatPromptTemplate
- 프롬프트 템플릿보다 더 자연스럽고 좋은 답변
- '대화'에 중점을 둠
- 튜플 형식으로 구성
- (role, message) 형태
- role: system, human, ai가 있음 (셋 중 하나 무조건 넣어야 함)
- system: 시스템 설정 메세지, 대화 전체에 적용되는 설정, 페르소나 지정 가능
- human: 사용자가 입력하는 매세지, 여러 개의 메세지를 포함할 수 있어 대화의 흐름 더욱 자연스럽게 구성 가능
- ai: AI의 응답 메세지 의미. 이전 대화 내용을 기반으로 생성된 답변 포함

In [48]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_template("{country}의 수도는 어디인가요?")
chat_prompt.format(country = '대한민국')

'Human: 대한민국의 수도는 어디인가요?'

In [51]:
from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate(
    [
        ('system', '당신은 친절한 AI 어시스턴트입니다. 당신의 이름은 {name}입니다.'),
        ('human', '반가워요!'),
        ('ai', '안녕하세요! 무엇을 도와드릴까요?'),
        ('human', '{user_input}'),
    ]
)

messages = chat_template.format_messages(
    name = '동호', user_input = '당신의 이름은 무엇입니까?'
)

llm = ChatOpenAI(
    temperature=0.2,
    model_name = 'gpt-4o-mini'
)
llm.invoke(messages).content

'제 이름은 동호입니다! 당신과 대화하게 되어 기쁩니다. 다른 질문이나 궁금한 점이 있으면 말씀해 주세요!'

In [52]:
chain = chat_template | llm
chain.invoke({'name': '동호', 'user_input': '당신의 이름은 무엇입니가?'}).content

'제 이름은 동호입니다! 당신은 어떻게 지내고 계신가요?'

MessagesPlaceholder
- 아직 확정된 메세지는 아니지만 나중에 채워질 메세지를 채우기 위해 임시로 확보한 자리
- 대화는 진행중에 쌓이지만 저장해 두었다가 나중에 대화기록 분석 및 다른 질문 활용 가능

In [1]:
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv()
logging.langsmith("CH02-Prompt")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH02-Prompt


In [2]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import MessagesPlaceholder, ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages(
    [
        (
            'system',
            '당신은 요약 전문 AI 어시스턴트입니다. 당신의 임무는 주요 키워드로 대화를 요약하는 것입니다.',
                            ),
    MessagesPlaceholder(variable_name='conversation'),
    ('human', '지금까지의 대화를 {word_count} 단어로 요약합니다.')

    ]
)
chat_prompt

ChatPromptTemplate(input_variables=['conversation', 'word_count'], input_types={'conversation': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annota

In [4]:
formatted_chat_prompt = chat_prompt.format(
    word_count = 5,
    conversation = [
        ('human', '안녕하세요! 저는 오늘 새로 입사한 동호입니다. 만나서 반갑습니다.'),
        ('ai', '반가워요! 앞으로 잘 부탁드립니다.')
    ],
)

print(formatted_chat_prompt)

System: 당신은 요약 전문 AI 어시스턴트입니다. 당신의 임무는 주요 키워드로 대화를 요약하는 것입니다.
Human: 안녕하세요! 저는 오늘 새로 입사한 동호입니다. 만나서 반갑습니다.
AI: 반가워요! 앞으로 잘 부탁드립니다.
Human: 지금까지의 대화를 5 단어로 요약합니다.


In [5]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI()

chain = chat_prompt | llm | StrOutputParser()

chain.invoke(
    {
        'word_count' : 5,
        'conversation' : [
            (
                'human',
                '안녕하세요! 저는 오늘 새로 입사한 동호입니다. 만나서 반갑습니다.',
            ),
            ('ai', '반가워요! 앞으로 잘 부탁드립니다.'),
        ],
    }
)

'새로 입사한 동호, 만나서 반가워요!'

FewShotPromptTemplete (퓨샷 기법)
- 예시를 제공하여 답변형식을 미리 알려줘 더 좋은 결과 처리
- 원 샷: 하나의 답변 예시를 제공, 퓨 샷: 두 개 이상의 단변 예시 제공

In [6]:
#딕셔너리 구조로 예시 질문 형식화

examples = [
    {
        "question": "스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 스티브 잡스는 몇 살에 사망했나요?
중간 답변: 스티브 잡스는 56세에 사망했습니다.
추가 질문: 아인슈타인은 몇 살에 사망했나요?
중간 답변: 아인슈타인은 76세에 사망했습니다.
최종 답변은: 아인슈타인
""",
    }
]

In [7]:
from langchain_core.prompts.few_shot import FewShotPromptTemplate
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

example_prompt = PromptTemplate.from_template(
    'Question:\n{question}\nAnswer\n{answer} '
)
print(example_prompt.format(**examples[0]))

Question:
스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?
Answer
이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 스티브 잡스는 몇 살에 사망했나요?
중간 답변: 스티브 잡스는 56세에 사망했습니다.
추가 질문: 아인슈타인은 몇 살에 사망했나요?
중간 답변: 아인슈타인은 76세에 사망했습니다.
최종 답변은: 아인슈타인
 


In [ ]:
prompt = FewShotPromptTemplate(
    examples=examples, #example 질문 양식 제공
    example_prompt=example_prompt, #example 질문 양식 format 형식
    suffix = 'Question:\n{question}\nAnswer:', #실제 질문(이때 question에 사용자 질문 추가 됨)
    input_variables = ['question'], #최종 완성까지 외부에서 새로 받아야 하는 변수 (사용자가 적는 question 한 개)
)

question= 'Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?'
final_prompt = prompt.format(question = question)
print(final_prompt)

Question:
스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?
Answer
이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 스티브 잡스는 몇 살에 사망했나요?
중간 답변: 스티브 잡스는 56세에 사망했습니다.
추가 질문: 아인슈타인은 몇 살에 사망했나요?
중간 답변: 아인슈타인은 76세에 사망했습니다.
최종 답변은: 아인슈타인
 

Question:
Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?
Answer:


In [10]:
from langchain_openai import ChatOpenAI

# 스트리밍 출력용 함수 정의
def stream_response(stream):
    for chunk in stream:
        print(chunk.content, end="", flush=True)
    print() # 스트리밍 끝난 후 줄바꿈

llm = ChatOpenAI(
    temperature=0,
    model_name='gpt-4o-mini',
)

# 실행
answer = llm.stream(final_prompt)
stream_response(answer)

Google은 1998년에 창립되었습니다. Bill Gates는 1955년 10월 28일에 태어났으므로, 1998년에는 42세였습니다.


In [15]:
#chain을 만들어 동적으로 질문을 입력받아 답변을 받음
#할루시네이션 현상을 줄이고 더 정확한 답변을 얻을 수 있음

from langchain_teddynote.messages import stream_response

prompt = FewShotPromptTemplate(
    examples = examples,
    example_prompt = example_prompt,
    suffix = 'Question:\n{question}\nAnswer:',
    input_variables = ['question'],
)

chain = prompt | llm | StrOutputParser()

answer = chain.stream(
    {'question': "Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?"}
)
stream_response(answer)

Google은 1998년에 창립되었습니다. Bill Gates는 1955년 10월 28일에 태어났으므로, 1998년에는 42세였습니다.

예제 선택기
- 예제를 사용하면 좋은 답변을 얻지만 비용이 많이 발생함
- 질문과 유사한 예시만 선택해 프롬프트에 넣는 방식
- SemanticSimilarityExampleSelector: 입력된 질문과 의미가 가장 유사한 예시 선택
- MaxMarginalRelevanceExampleSelector: 유사도 뿐만 아니라 예시의 다양성까지 고려하여 선택

In [19]:
#OpenAIEmbeddings: 질문이나 텍스트를 벡터로 변환하기 위해 임베딩을 생성하는 클래스
#Chroma: 벡터 스토어 데이터베이스, 예제를 저장해 두고 사용자의 질문이 들어오면 저장된 예시와 유사도 계산을 통해 가장 적합한 예제 선택

from langchain_core.example_selectors import (
    MaxMarginalRelevanceExampleSelector,
    SemanticSimilarityExampleSelector
)
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma


examples = [
    {
        "question": "스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 스티브 잡스는 몇 살에 사망했나요?
중간 답변: 스티브 잡스는 56세에 사망했습니다.
추가 질문: 아인슈타인은 몇 살에 사망했나요?
중간 답변: 아인슈타인은 76세에 사망했습니다.
최종 답변은: 아인슈타인
""",
    },
    {
        "question": "네이버의 창립자는 언제 태어났나요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 네이버의 창립자는 누구인가요?
중간 답변: 네이버는 이해진에 의해 창립되었습니다.
추가 질문: 이해진은 언제 태어났나요?
중간 답변: 이해진은 1967년 6월 22일에 태어났습니다.
최종 답변은: 1967년 6월 22일
""",
    },
    {
        "question": "율곡 이이의 어머니가 태어난 해의 통치하던 왕은 누구인가요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 율곡 이이의 어머니는 누구인가요?
중간 답변: 율곡 이이의 어머니는 신사임당입니다.
추가 질문: 신사임당은 언제 태어났나요?
중간 답변: 신사임당은 1504년에 태어났습니다.
추가 질문: 1504년에 조선을 통치한 왕은 누구인가요?
중간 답변: 1504년에 조선을 통치한 왕은 연산군입니다.
최종 답변은: 연산군
""",
    },
    {
        "question": "올드보이와 기생충의 감독이 같은 나라 출신인가요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 올드보이의 감독은 누구인가요?
중간 답변: 올드보이의 감독은 박찬욱입니다.
추가 질문: 박찬욱은 어느 나라 출신인가요?
중간 답변: 박찬욱은 대한민국 출신입니다.
추가 질문: 기생충의 감독은 누구인가요?
중간 답변: 기생충의 감독은 봉준호입니다.
추가 질문: 봉준호는 어느 나라 출신인가요?
중간 답변: 봉준호는 대한민국 출신입니다.
최종 답변은: 예
""",
    },
]

chroma = Chroma(
    collection_name='example_selector',
    embedding_function=OpenAIEmbeddings(),
)

example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples,
    OpenAIEmbeddings(),
    Chroma,
    k=1, #예시의 개수 1개
)

In [20]:
#select_examples(): 질문과 의미적으로 가장 유사한 하나의 예시 선택 후 저장

selected_examples = example_selector.select_examples({'question': question})

question = 'Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?'
print(f"입력에 가장 유사한 예시: \n{question}\n")

for example in selected_examples:
    print(f'question:\n{example['question']}')
    print(f'answer:\n{example['answer']}')

입력에 가장 유사한 예시: 
Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?

question:
네이버의 창립자는 언제 태어났나요?
answer:
이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 네이버의 창립자는 누구인가요?
중간 답변: 네이버는 이해진에 의해 창립되었습니다.
추가 질문: 이해진은 언제 태어났나요?
중간 답변: 이해진은 1967년 6월 22일에 태어났습니다.
최종 답변은: 1967년 6월 22일



In [21]:
prompt = FewShotPromptTemplate(
    example_selector = example_selector,
    example_prompt = example_prompt,
    suffix="Question:\n{question}\nAnswer:",
    input_variables=['question'],
)

question = 'Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?'
example_selector_prompt = prompt.format(question = question)
print(example_selector_prompt)

Question:
네이버의 창립자는 언제 태어났나요?
Answer
이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 네이버의 창립자는 누구인가요?
중간 답변: 네이버는 이해진에 의해 창립되었습니다.
추가 질문: 이해진은 언제 태어났나요?
중간 답변: 이해진은 1967년 6월 22일에 태어났습니다.
최종 답변은: 1967년 6월 22일
 

Question:
Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?
Answer:


In [22]:
prompt = FewShotPromptTemplate(
    example_selector = example_selector,
    example_prompt = example_prompt,
    suffix="Question:\n{question}\nAnswer:",
    input_variables=['question'],
)

chain = prompt | llm
answer = chain.stream(
    {'question': "Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?"}
)
stream_response(answer)

Google은 1998년에 창립되었습니다. Bill Gates는 1955년 10월 28일에 태어났으므로, 1998년에는 42세였습니다.

MMR(Maximal Marginal Relevance) 알고리즘
- 정보 검색과 문서 요약을 할 때, 다양성과 관련성을 모두 고려하여 결과를 선택하는 방법
- 비슷한 내용이 반복되지 않도록 하면서도 사용자가 찾고 있는 질문과 가장 잘 맞는 정보 제공
    (1) 관련성(relavance): 사용자가 입력한 검색어나 주제와 문서가 얼마나 잘 맞는지를 평가하는 기준. 문서에 검색 내용과 얼마나 잘 일치하는지 점수를 매기고 높은 점수의 문서가 선택됨
    (2) 다양성(diversity): 이미 선택된 문서와 다른 새로운 문서 간의 유사성을 계산해서, 이미 선택된 문서와 비슷한 내용의 문서는 다시 선택되지 않도록 함
- 람다의 값에 의해 조절 (람다 값이 클수록 관련성 중시, 람다 값이 작을 수록 다양성 중시)

FewshotChatMessagePromptTemplate
- ChatMessage가 추가됨

In [23]:
#(1. 회의록, 2.요약, 3. 문장 교정)을 하는 전문가 역할 기반

examples = [
    {
        "instruction": "당신은 회의록 작성 전문가 입니다. 주어진 정보를 바탕으로 회의록을 작성해 주세요",
        "input": "2023년 12월 25일, XYZ 회사의 마케팅 전략 회의가 오후 3시에 시작되었다. 회의에는 마케팅 팀장인 김수진, 디지털 마케팅 담당자인 박지민, 소셜 미디어 관리자인 이준호가 참석했다. 회의의 주요 목적은 2024년 상반기 마케팅 전략을 수립하고, 새로운 소셜 미디어 캠페인에 대한 아이디어를 논의하는 것이었다. 팀장인 김수진은 최근 시장 동향에 대한 간략한 개요를 제공했으며, 이어서 각 팀원이 자신의 분야에서의 전략적 아이디어를 발표했다.",
        "answer": """
회의록: XYZ 회사 마케팅 전략 회의
일시: 2023년 12월 25일
장소: XYZ 회사 회의실
참석자: 김수진 (마케팅 팀장), 박지민 (디지털 마케팅 담당자), 이준호 (소셜 미디어 관리자)

1. 개회
   - 회의는 김수진 팀장의 개회사로 시작됨.
   - 회의의 목적은 2024년 상반기 마케팅 전략 수립 및 새로운 소셜 미디어 캠페인 아이디어 논의.

2. 시장 동향 개요 (김수진)
   - 김수진 팀장은 최근 시장 동향에 대한 분석을 제시.
   - 소비자 행동 변화와 경쟁사 전략에 대한 통찰 공유.

3. 디지털 마케팅 전략 (박지민)
   - 박지민은 디지털 마케팅 전략에 대해 발표.
   - 온라인 광고와 SEO 최적화 방안에 중점을 둠.

4. 소셜 미디어 캠페인 (이준호)
   - 이준호는 새로운 소셜 미디어 캠페인에 대한 아이디어를 제안.
   - 인플루언서 마케팅과 콘텐츠 전략에 대한 계획을 설명함.

5. 종합 논의
   - 팀원들 간의 아이디어 공유 및 토론.
   - 각 전략에 대한 예산 및 자원 배분에 대해 논의.

6. 마무리
   - 다음 회의 날짜 및 시간 확정.
   - 회의록 정리 및 배포는 박지민 담당.
""",
    },
    {
        "instruction": "당신은 요약 전문가 입니다. 다음 주어진 정보를 바탕으로 내용을 요약해 주세요",
        "input": "이 문서는 '지속 가능한 도시 개발을 위한 전략'에 대한 20페이지 분량의 보고서입니다. 보고서는 지속 가능한 도시 개발의 중요성, 현재 도시화의 문제점, 그리고 도시 개발을 지속 가능하게 만들기 위한 다양한 전략을 포괄적으로 다루고 있습니다. 이 보고서는 또한 성공적인 지속 가능한 도시 개발 사례를 여러 국가에서 소개하고, 이러한 사례들을 통해 얻은 교훈을 요약하고 있습니다.",
        "answer": """
문서 요약: 지속 가능한 도시 개발을 위한 전략 보고서

- 중요성: 지속 가능한 도시 개발이 필수적인 이유와 그에 따른 사회적, 경제적, 환경적 이익을 강조.
- 현 문제점: 현재의 도시화 과정에서 발생하는 주요 문제점들, 예를 들어 환경 오염, 자원 고갈, 불평등 증가 등을 분석.
- 전략: 지속 가능한 도시 개발을 달성하기 위한 다양한 전략 제시. 이에는 친환경 건축, 대중교통 개선, 에너지 효율성 증대, 지역사회 참여 강화 등이 포함됨.
- 사례 연구: 전 세계 여러 도시의 성공적인 지속 가능한 개발 사례를 소개. 예를 들어, 덴마크의 코펜하겐, 일본의 요코하마 등의 사례를 통해 실현 가능한 전략들을 설명.
- 교훈: 이러한 사례들에서 얻은 주요 교훈을 요약. 강조된 교훈에는 다각적 접근의 중요성, 지역사회와의 협력, 장기적 계획의 필요성 등이 포함됨.

이 보고서는 지속 가능한 도시 개발이 어떻게 현실적이고 효과적인 형태로 이루어질 수 있는지에 대한 심도 있는 분석을 제공합니다.
""",
    },
    {
        "instruction": "당신은 문장 교정 전문가 입니다. 다음 주어진 문장을 교정해 주세요",
        "input": "우리 회사는 새로운 마케팅 전략을 도입하려고 한다. 이를 통해 고객과의 소통이 더 효과적이 될 것이다.",
        "answer": "본 회사는 새로운 마케팅 전략을 도입함으로써, 고객과의 소통을 보다 효과적으로 개선할 수 있을 것으로 기대된다.",
    },
]

In [24]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

chroma = Chroma('fewshot_chat', OpenAIEmbeddings())

example_prompt = ChatPromptTemplate.from_messages(
    [
        ('human', '{instruction}:\n{input}'),
        ('ai','{answer}'),
    ]
)

example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples, OpenAIEmbeddings(), chroma, k=1,
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt
)

In [25]:
question = {
    "instruction": "회의록을 작성해 주세요",
    "input": "2023년 12월 26일, ABC 기술 회사의 제품 개발 팀은 새로운 모바일 애플리케이션 프로젝트에 대한 주간 진행 상황 회의를 가졌다. 이 회의에는 프로젝트 매니저인 최현수, 주요 개발자인 황지연, UI/UX 디자이너인 김태영이 참석했다. 회의의 주요 목적은 프로젝트의 현재 진행 상황을 검토하고, 다가오는 마일스톤에 대한 계획을 수립하는 것이었다. 각 팀원은 자신의 작업 영역에 대한 업데이트를 제공했고, 팀은 다음 주까지의 목표를 설정했다.",
}

example_selector.select_examples(question)

[{'instruction': '당신은 회의록 작성 전문가 입니다. 주어진 정보를 바탕으로 회의록을 작성해 주세요',
  'answer': '\n회의록: XYZ 회사 마케팅 전략 회의\n일시: 2023년 12월 25일\n장소: XYZ 회사 회의실\n참석자: 김수진 (마케팅 팀장), 박지민 (디지털 마케팅 담당자), 이준호 (소셜 미디어 관리자)\n\n1. 개회\n   - 회의는 김수진 팀장의 개회사로 시작됨.\n   - 회의의 목적은 2024년 상반기 마케팅 전략 수립 및 새로운 소셜 미디어 캠페인 아이디어 논의.\n\n2. 시장 동향 개요 (김수진)\n   - 김수진 팀장은 최근 시장 동향에 대한 분석을 제시.\n   - 소비자 행동 변화와 경쟁사 전략에 대한 통찰 공유.\n\n3. 디지털 마케팅 전략 (박지민)\n   - 박지민은 디지털 마케팅 전략에 대해 발표.\n   - 온라인 광고와 SEO 최적화 방안에 중점을 둠.\n\n4. 소셜 미디어 캠페인 (이준호)\n   - 이준호는 새로운 소셜 미디어 캠페인에 대한 아이디어를 제안.\n   - 인플루언서 마케팅과 콘텐츠 전략에 대한 계획을 설명함.\n\n5. 종합 논의\n   - 팀원들 간의 아이디어 공유 및 토론.\n   - 각 전략에 대한 예산 및 자원 배분에 대해 논의.\n\n6. 마무리\n   - 다음 회의 날짜 및 시간 확정.\n   - 회의록 정리 및 배포는 박지민 담당.\n',
  'input': '2023년 12월 25일, XYZ 회사의 마케팅 전략 회의가 오후 3시에 시작되었다. 회의에는 마케팅 팀장인 김수진, 디지털 마케팅 담당자인 박지민, 소셜 미디어 관리자인 이준호가 참석했다. 회의의 주요 목적은 2024년 상반기 마케팅 전략을 수립하고, 새로운 소셜 미디어 캠페인에 대한 아이디어를 논의하는 것이었다. 팀장인 김수진은 최근 시장 동향에 대한 간략한 개요를 제공했으며, 이어서 각 팀원이 자신의 분야에서의 전략적 아이디어를 발표했다.'}]

In [27]:
final_prompt = ChatPromptTemplate.from_messages(
    [
        ('system', 'You are a helpful assistant'),
        few_shot_prompt,
        ('human', '{instruction}\n{input}'),
    ]
)

chain = final_prompt | llm
answer = chain.stream(question)
stream_response(answer)

회의록: ABC 기술 회사 제품 개발 팀 주간 진행 상황 회의  
일시: 2023년 12월 26일  
장소: ABC 기술 회사 회의실  
참석자: 최현수 (프로젝트 매니저), 황지연 (주요 개발자), 김태영 (UI/UX 디자이너)  

1. 개회  
   - 회의는 최현수 프로젝트 매니저의 개회사로 시작됨.  
   - 회의의 목적은 새로운 모바일 애플리케이션 프로젝트의 현재 진행 상황 검토 및 다가오는 마일스톤 계획 수립.

2. 진행 상황 업데이트  
   - **최현수 (프로젝트 매니저)**  
     - 전체 프로젝트 일정 및 마일스톤에 대한 개요 제공.  
     - 현재 진행 상황이 계획에 부합하고 있음을 확인.  

   - **황지연 (주요 개발자)**  
     - 개발 진행 상황 보고.  
     - 주요 기능 구현 완료 및 버그 수정 작업 진행 중.  
     - 다음 주까지의 개발 목표 설정.  

   - **김태영 (UI/UX 디자이너)**  
     - 디자인 프로토타입 및 사용자 피드백 결과 공유.  
     - UI 개선 사항 및 사용자 경험 향상을 위한 제안 발표.  

3. 목표 설정  
   - 팀원들은 다음 주까지의 목표를 설정하고 각자의 작업 계획을 조율.  
   - 마일스톤에 맞춰 필요한 리소스 및 지원 요청 사항 논의.

4. 종합 논의  
   - 팀원 간의 의견 교환 및 추가적인 아이디어 공유.  
   - 프로젝트 진행에 있어 발생할 수 있는 잠재적 문제점에 대한 논의.

5. 마무리  
   - 다음 주 회의 일정 및 준비 사항 확인.  
   - 회의록 작성 및 배포는 최현수 담당.  
   - 회의 종료.

CustomExampleSelector

- instuction만 사용하여 검색했을 때, instruction으로 따졌을 때는 유사하지 않은 예시임에도 input값과 비슷하면 유사한 예시로 판단할 수 있음
- 이를 해결하기 위해 커스텀 유사도 계산을 수행하는 변도의 커스텀 예제 선택기 클래스 사용

Langchain-Hub 프롬포트 불러오기
- QA Over dociments의 rlm/rag-prompt 불러오기
- 각각의 커밋 해쉬를 사용해서 버젼 별 선택 가능
- 자신의 프롬포트도 업로드 가능

In [6]:
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv()
logging.langsmith("CH02-Prompt")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH02-Prompt


In [17]:
from langsmith import Client

client = Client()

# 공개 프롬프트 수락 옵션(dangerously_pull_public_prompt=True) 추가
prompt = client.pull_prompt("rlm/rag-prompt", dangerously_pull_public_prompt=True)

print(prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]


In [19]:
#내 프롬포트 push

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    '주어진 내용을 바탕으로 다음 문장을 요약하세요. 답변은 반드시 한글로 작성하게요\n\nCONTEXT: {content}\n\nSUMMARY: '
)
prompt

ChatPromptTemplate(input_variables=['content'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['content'], input_types={}, partial_variables={}, template='주어진 내용을 바탕으로 다음 문장을 요약하세요. 답변은 반드시 한글로 작성하게요\n\nCONTEXT: {content}\n\nSUMMARY: '), additional_kwargs={})])

In [22]:
from langsmith import Client

prompt_url = client.push_prompt(
    "simple-summary-prompt", 
    object=prompt,
    is_public=False  # True로 설정하면 전체 공개, False는 비공개(Private)
)

print(f"업로드 완료! URL: {prompt_url}")

업로드 완료! URL: https://smith.langchain.com/prompts/simple-summary-prompt/7312213c?organizationId=b729d7d3-1598-4daa-8af7-cb3d01808b22
